In [1]:
import os
import random
import shutil
import numpy as np
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, WeightedRandomSampler

from PIL import Image

In [2]:
import kagglehub

path = kagglehub.dataset_download("kacpergregorowicz/house-plant-species")
print("Dataset path:", path)

Dataset path: C:\Users\1alma\.cache\kagglehub\datasets\kacpergregorowicz\house-plant-species\versions\4


In [3]:
def is_valid_image(path):
    try:
        img = Image.open(path)
        img.verify()
        return True
    except:
        return False

In [4]:
def split_dataset(src_dir, dst_dir, val_ratio=0.2):
    train_dir = os.path.join(dst_dir, "train")
    val_dir = os.path.join(dst_dir, "val")

    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(val_dir, exist_ok=True)

    for class_name in os.listdir(src_dir):
        class_path = os.path.join(src_dir, class_name)

        if not os.path.isdir(class_path):
            continue

        images = [img for img in os.listdir(class_path)
                  if is_valid_image(os.path.join(class_path, img))]

        random.shuffle(images)

        split_idx = int(len(images) * (1 - val_ratio))

        train_imgs = images[:split_idx]
        val_imgs = images[split_idx:]

        os.makedirs(os.path.join(train_dir, class_name), exist_ok=True)
        os.makedirs(os.path.join(val_dir, class_name), exist_ok=True)

        for img in train_imgs:
            shutil.copy(os.path.join(class_path, img),
                        os.path.join(train_dir, class_name, img))

        for img in val_imgs:
            shutil.copy(os.path.join(class_path, img),
                        os.path.join(val_dir, class_name, img))

    return train_dir, val_dir

In [5]:
src_dir = os.path.join(path, "house_plant_species")
data_dir = os.path.join(os.getcwd(), "processed")

train_dir, val_dir = split_dataset(src_dir, data_dir)

print(train_dir, val_dir)

C:\Users\1alma\DataspellProjects\PlantGuard_AI\processed\train C:\Users\1alma\DataspellProjects\PlantGuard_AI\processed\val


In [6]:
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(300),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.GaussianBlur(3),
    transforms.ToTensor(),
])

val_transforms = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
])

In [7]:
train_dataset = datasets.ImageFolder(train_dir, transform=train_transforms)
val_dataset = datasets.ImageFolder(val_dir, transform=val_transforms)

class_names = train_dataset.classes
num_classes = len(class_names)

print("Classes:", num_classes)

Classes: 47


In [8]:
targets = [label for _, label in train_dataset.samples]
class_count = Counter(targets)

class_weights = 1. / torch.tensor(
    [class_count[i] for i in range(num_classes)],
    dtype=torch.float
)

weights = [class_weights[label] for label in targets]

sampler = WeightedRandomSampler(
    weights=weights,
    num_samples=len(weights),
    replacement=True
)

In [9]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    sampler=sampler,
    num_workers=4
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4
)

In [10]:
model = models.efficientnet_b3(weights="IMAGENET1K_V1")

in_features = model.classifier[1].in_features

model.classifier = nn.Sequential(
    nn.Linear(in_features, 512),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(512, num_classes)
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [11]:
criterion = nn.CrossEntropyLoss(
    weight=class_weights.to(device),
    label_smoothing=0.1
)

optimizer = optim.AdamW(model.parameters(), lr=1e-3)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=10
)

In [12]:
for param in model.features.parameters():
    param.requires_grad = False

In [13]:
from tqdm import tqdm

def train_epoch(model, loader, optimizer, criterion):
    model.train()

    total_loss = 0
    correct = 0

    loop = tqdm(loader, desc="Training", leave=False)

    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()

        loop.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(loader)
    acc = correct / len(loader.dataset)

    return avg_loss, acc

In [14]:
def validate(model, loader, criterion):
    model.eval()

    total_loss = 0
    correct = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()

    avg_loss = total_loss / len(loader)
    acc = correct / len(loader.dataset)

    return avg_loss, acc

In [15]:
best_acc = 0
patience = 5
counter = 0

In [16]:
train_losses = []
val_losses = []
train_accs = []
val_accs = []
lrs = []

In [18]:
epochs = 10

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")

    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = validate(model, val_loader, criterion)

    scheduler.step()

    current_lr = optimizer.param_groups[0]['lr']

    # логируем
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    train_accs.append(train_acc)
    val_accs.append(val_acc)

    lrs.append(current_lr)

    print(f"LR: {current_lr:.6f}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    # early stopping
    if val_acc > best_acc:
        best_acc = val_acc
        counter = 0
        torch.save(model.state_dict(), "best_model.pth")
        print(" Best model saved")
    else:
        counter += 1
        print(f"No improvement ({counter}/{patience})")

        if counter >= patience:
            print("Early stopping triggered")
            break

LR: 0.000000
Train Loss: 1.4053 | Train Acc: 0.7671
Val Loss: 3.9950 | Val Acc: 0.7304
No improvement (1/5)


In [17]:
for param in model.features.parameters():
    param.requires_grad = True

optimizer = optim.AdamW(model.parameters(), lr=1e-4)

In [18]:
import matplotlib.pyplot as plt

def plot_training_history():
    epochs_range = range(1, len(train_losses) + 1)

    plt.figure(figsize=(16, 5))

    # LOSS
    plt.subplot(1, 3, 1)
    plt.plot(epochs_range, train_losses, marker='o', label="Train Loss")
    plt.plot(epochs_range, val_losses, marker='o', label="Val Loss")
    plt.title("Loss over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid()

    # ACCURACY
    plt.subplot(1, 3, 2)
    plt.plot(epochs_range, train_accs, marker='o', label="Train Acc")
    plt.plot(epochs_range, val_accs, marker='o', label="Val Acc")
    plt.title("Accuracy over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid()

    # LR
    plt.subplot(1, 3, 3)
    plt.plot(epochs_range, lrs, marker='o')
    plt.title("Learning Rate Schedule")
    plt.xlabel("Epoch")
    plt.ylabel("LR")
    plt.grid()

    plt.tight_layout()
    plt.show()

In [19]:
def analyze_training():
    best_epoch = np.argmax(val_accs) + 1
    print(f"Best epoch: {best_epoch}")
    print(f"Best Val Acc: {max(val_accs):.4f}")

    if train_accs[-1] > val_accs[-1] + 0.05:
        print("Possible overfitting detected")

    if val_accs[-1] < 0.7:
        print("Model may be underfitting")

GradCam

In [20]:
import torch
import torch.nn.functional as F
import cv2
import numpy as np

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer

        self.activations = None

        self._handles = []

        self._register_hooks()

    def remove_hooks(self):
        for handle in self._handles:
            try:
                handle.remove()
            except Exception:
                pass
        self._handles = []

    def _register_hooks(self):
        def forward_hook(module, input, output):
            if isinstance(output, (tuple, list)):
                output = output[0]
            self.activations = output

        self._handles.append(self.target_layer.register_forward_hook(forward_hook))

    def generate(self, input_tensor, class_idx=None, return_class_idx=False):
        self.model.eval()

        self.activations = None

        with torch.enable_grad():
            output = self.model(input_tensor)

        if class_idx is None:
            class_idx = torch.argmax(output, dim=1).item()

        if self.activations is None:
            raise RuntimeError("Grad-CAM forward hook did not capture activations; check target_layer")

        self.model.zero_grad(set_to_none=True)
        score = output[:, class_idx].sum()
        grads = torch.autograd.grad(score, self.activations, retain_graph=False, create_graph=False)[0]

        gradients = grads[0]  # [C, H, W]
        activations = self.activations[0].detach()  # [C, H, W]

        weights = torch.mean(gradients, dim=(1, 2))  # [C]

        cam = (weights[:, None, None] * activations).sum(dim=0)

        cam = F.relu(cam)

        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)

        cam = cam.detach().cpu().numpy()
        if return_class_idx:
            return cam, class_idx
        return cam

In [21]:
def overlay_cam(image_rgb, cam, alpha=0.4, colormap=cv2.COLORMAP_JET):
    if cam.ndim != 2:
        cam = np.squeeze(cam)

    cam = cv2.resize(cam, (image_rgb.shape[1], image_rgb.shape[0]))
    cam = np.clip(cam, 0.0, 1.0)
    heatmap_bgr = cv2.applyColorMap(np.uint8(255 * cam), colormap)

    if image_rgb.ndim == 2:
        image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_GRAY2BGR)
    else:
        image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)

    overlay_bgr = cv2.addWeighted(image_bgr, 1 - alpha, heatmap_bgr, alpha, 0)
    overlay_rgb = cv2.cvtColor(overlay_bgr, cv2.COLOR_BGR2RGB)
    return overlay_rgb

In [22]:
target_layer = model.features[-1]
grad_cam = GradCAM(model, target_layer)

In [23]:
def visualize_gradcam(
    model,
    grad_cam,
    image_tensor,
    class_names,
    device=None,
    mean=(0.485, 0.456, 0.406),
    std=(0.229, 0.224, 0.225),
    debug=False,
):
    model.eval()

    if device is None:
        device = next(model.parameters()).device

    input_tensor = image_tensor.unsqueeze(0).to(device)
    cam, pred = grad_cam.generate(input_tensor, return_class_idx=True)

    if debug:
        print("cam:", getattr(cam, "shape", None), float(np.min(cam)), float(np.max(cam)))

    img = image_tensor.detach().cpu().float().permute(1, 2, 0).numpy()
    if mean is not None and std is not None:
        mean_arr = np.asarray(mean, dtype=np.float32)
        std_arr = np.asarray(std, dtype=np.float32)
        img = img * std_arr + mean_arr
    img = np.clip(img, 0.0, 1.0)
    img_u8 = (img * 255).astype(np.uint8)

    overlay = overlay_cam(img_u8, cam)

    if debug:
        print("img:", img_u8.shape, int(img_u8.min()), int(img_u8.max()), "pred:", pred)

    import matplotlib.pyplot as plt

    plt.figure(figsize=(8, 4))

    plt.subplot(1, 2, 1)
    plt.imshow(img_u8)
    plt.title("Original")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(overlay)
    title = class_names[pred] if pred < len(class_names) else str(pred)
    plt.title(f"Grad-CAM: {title}")
    plt.axis("off")

    plt.show()

Visualization metrics

In [24]:
def get_predictions(model, loader):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)

            outputs = model(images)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    return np.array(all_preds), np.array(all_labels)

In [ ]:
y_pred, y_true = get_predictions(model, val_loader)

KeyboardInterrupt: 

In [28]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

acc = accuracy_score(y_true, y_pred)

precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="weighted"
)

print(f"Accuracy: {acc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")

Accuracy: 0.7304
Precision: 0.7687
Recall: 0.7304
F1-score: 0.7345


In [26]:
from sklearn.metrics import precision_recall_fscore_support
import pandas as pd

precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, average=None
)

metrics_df = pd.DataFrame({
    "Class": class_names,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
    "Support": support
}).sort_values(by="F1")

NameError: name 'y_true' is not defined

In [1]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 16))

sns.heatmap(
    metrics_df.set_index("Class")[["Precision", "Recall", "F1"]],
    annot=True,
    fmt=".2f",
    cmap="RdYlGn",
    vmin=0,
    vmax=1
)

plt.title("Per-Class Metrics Heatmap")
plt.show()

<Figure size 1000x1600 with 0 Axes>

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(14, 12))
sns.heatmap(
    cm,
    xticklabels=class_names,
    yticklabels=class_names,
    cmap="Blues"
)
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.title("Confusion Matrix")
plt.show()

In [ ]:
errors = y_pred != y_true
misclassified = list(zip(y_true[errors], y_pred[errors]))

from collections import Counter
error_counts = Counter(misclassified)

for (true, pred), count in error_counts.most_common(10):
    print(f"{class_names[true]} → {class_names[pred]}: {count}")

In [ ]:
def show_misclassified(model, loader, class_names, num_images=6):
    model.eval()
    shown = 0

    plt.figure(figsize=(12, 8))

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            for i in range(len(images)):
                if preds[i] != labels[i]:
                    shown += 1

                    plt.subplot(2, 3, shown)
                    img = images[i].cpu().permute(1, 2, 0)
                    plt.imshow(img)
                    plt.title(f"T: {class_names[labels[i]]}\nP: {class_names[preds[i]]}")
                    plt.axis("off")

                    if shown == num_images:
                        plt.show()
                        return

In [ ]:
def top_k_accuracy(model, loader, k=3):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)

            outputs = model(images)
            topk = torch.topk(outputs, k=k, dim=1).indices.cpu()

            for i in range(len(labels)):
                if labels[i] in topk[i]:
                    correct += 1
                total += 1

    return correct / total

print("Top-3 Accuracy:", top_k_accuracy(model, val_loader, k=3))

In [ ]:
class_counts = Counter(y_true)

plt.figure(figsize=(12, 6))
plt.bar(class_names, [class_counts[i] for i in range(len(class_names))])
plt.xticks(rotation=90)
plt.title("Class Distribution")
plt.show()